# Lecture 5 Examples and Case — CSV and pandas Inspection

**Course:** AAU E26 — Introduction to Scripting, Data Mining and Machine Learning  
**Lecture:** Lecture 5  
**Goal:** Load and inspect the recurring monthly service report without cleaning it prematurely.

[Open this notebook in Google Colab](https://colab.research.google.com/github/asmrabbi/E26_TAN7_Scripting_CPH/blob/main/notebooks/examples/L05_examples_csv_pandas_inspection.ipynb) · [View the course repository](https://github.com/asmrabbi/E26_TAN7_Scripting_CPH)

Run the cells from top to bottom. Every executable line includes a short comment explaining what it does.


## Goal

Load a CSV file with pandas, inspect its structure, select observations, and write an initial data-quality note.


## Setup

The notebook loads the course dataset locally during validation and from GitHub in Google Colab.

### Running this notebook on a local computer

1. Download or clone the course repository.
2. Open a terminal in the repository folder.
3. Create and activate a virtual environment.
4. Install the shared requirements with `python -m pip install -r requirements.txt`.
5. Start Jupyter with `python -m jupyter lab`.

Packages used directly in this notebook: `pandas`

## Steps

### 1. Load the monthly service report


In [1]:
from pathlib import Path  # Imports Path so the notebook can find a local course file when available.
import pandas as pd  # Imports pandas for reading and working with table-shaped data.
remote_data_url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/monthly_service_report.csv"  # Stores the public GitHub address used by Google Colab.
local_data_candidates = [Path("data/monthly_service_report.csv"), Path("../../data/monthly_service_report.csv")]  # Lists possible local paths used during validation.
data_source = next((path for path in local_data_candidates if path.exists()), remote_data_url)  # Chooses a local file when present and otherwise uses GitHub.
service_data = pd.read_csv(data_source)  # Reads the CSV file into a pandas DataFrame.
print(service_data.head())  # Prints a small preview so we can confirm that loading worked.


   record_id report_month          city service_type  cases_received  \
0       1001   2026-01-01    Copenhagen      Housing             120   
1       1002   2026-01-01   copenhagen     Transport              85   
2       1003   2026-01-01       AALBORG      Housing              -3   
3       1004   2026-02-01    Koebenhavn   Employment              74   
4       1004   2026-02-01    Koebenhavn   Employment              74   

   cases_resolved resolution_days  satisfaction_score  \
0             112             5.1                 4.2   
1              90             3.2                 4.6   
2               0             8.4                 3.1   
3              68             4.0                 4.0   
4              68             4.0                 4.0   

                                  feedback  
0           Helpful staff and clear answer  
1  Quick answer but the form was confusing  
2                  Long wait for an answer  
3                   The guidance was clear  

### 2. Inspect shape, columns, and data types


In [2]:
print(service_data.shape)  # Displays the number of rows and columns.
print(service_data.columns.tolist())  # Displays every column name as a Python list.
print(service_data.dtypes)  # Displays the data type pandas inferred for each column.
service_data.info()  # Prints a compact summary of non-missing values and inferred types.


(16, 9)
['record_id', 'report_month', 'city', 'service_type', 'cases_received', 'cases_resolved', 'resolution_days', 'satisfaction_score', 'feedback']
record_id               int64
report_month              str
city                      str
service_type              str
cases_received          int64
cases_resolved          int64
resolution_days           str
satisfaction_score    float64
feedback                  str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   record_id           16 non-null     int64  
 1   report_month        16 non-null     str    
 2   city                16 non-null     str    
 3   service_type        16 non-null     str    
 4   cases_received      16 non-null     int64  
 5   cases_resolved      16 non-null     int64  
 6   resolution_days     16 non-null     str    
 7   satisfaction_score  15 non-n

### 3. Select useful columns and rows


In [3]:
selected_columns = service_data[["record_id", "city", "service_type", "cases_received"]]  # Keeps four columns relevant to the question.
copenhagen_rows = selected_columns[selected_columns["city"].astype(str).str.strip().str.lower() == "copenhagen"]  # Filters spelling-normalised Copenhagen rows.
print(copenhagen_rows.head())  # Displays a bounded preview of the selected observations.


    record_id          city service_type  cases_received
0        1001    Copenhagen      Housing             120
1        1002   copenhagen     Transport              85
6        1006    Copenhagen      Housing             131
8        1008    Copenhagen    Transport             105
10       1010    Copenhagen   Employment               0


### 4. Count visible quality concerns


In [4]:
missing_by_column = service_data.isna().sum()  # Counts missing values separately for every column.
duplicate_row_count = service_data.duplicated().sum()  # Counts rows that are exact duplicates of earlier rows.
unique_city_labels = sorted(service_data["city"].dropna().astype(str).unique().tolist())  # Lists the distinct raw city labels for consistency review.
print(missing_by_column[missing_by_column > 0])  # Displays only columns that contain at least one missing value.
print(f"Exact duplicate rows: {duplicate_row_count}")  # Displays the duplicate-row count.
print(unique_city_labels)  # Displays the raw city labels so inconsistent spelling or spacing is visible.


satisfaction_score    1
feedback              1
dtype: int64
Exact duplicate rows: 1
[' copenhagen ', 'AALBORG', 'Aalborg', 'Copenhagen', 'Koebenhavn']


### 5. Create an initial data note


In [5]:
data_note = {"source": "synthetic AAU teaching dataset", "unit_of_analysis": "one service category in one city and month", "rows": len(service_data), "quality_concerns": ["missing values", "inconsistent city labels", "mixed numeric formats", "duplicate rows"]}  # Records essential source and quality context.
for note_key, note_value in data_note.items():  # Visits each labelled part of the data note.
    print(f"{note_key}: {note_value}")  # Displays one documented field at a time.


source: synthetic AAU teaching dataset
unit_of_analysis: one service category in one city and month
rows: 16
quality_concerns: ['missing values', 'inconsistent city labels', 'mixed numeric formats', 'duplicate rows']


## Checks

Confirm that the raw city labels are inconsistent, at least one exact duplicate exists, and some numeric-looking columns have an `object` type because they contain non-numeric text.


## Next Steps

Do not clean the data yet. Use the exercise notebook to practise inspection and record what you can observe directly.
